In [12]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd
import pygris

print("=" * 70)
print("Cleaned ACS + Tract Join (with proper data types)")
print("=" * 70)

# ============================================================
# 1. LOAD CENSUS TRACT GEOMETRIES
# ============================================================

ia_tracts = pygris.tracts(state="IA", year=2024, cb=True)
ok_tracts = pygris.tracts(state="OK", year=2024, cb=True)
tx_tracts = pygris.tracts(state="TX", year=2024, cb=True)

tracts = pd.concat([ia_tracts, ok_tracts, tx_tracts], ignore_index=True)
print(f"Total census tracts loaded: {len(tracts):,}")

# ============================================================
# 2. LOAD ACS CSV FILES + CLEAN DATA TYPES
# ============================================================

# Helper function to safely convert columns to numeric
def clean_numeric(df, col):
    """Convert column to numeric, coercing errors to NaN"""
    df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Load files (update paths if your filenames are different)
acs_pop     = pd.read_csv("data/external/ACSDT5Y2024.B01003.csv")
acs_housing = pd.read_csv("data/external/ACSDT5Y2024.B25001.csv")
acs_income  = pd.read_csv("data/external/ACSDT5Y2024.B19013.csv")
acs_value   = pd.read_csv("data/external/ACSDT5Y2024.B25077.csv")

# Clean and rename columns
acs_pop = clean_numeric(acs_pop, "B01003_001E")
acs_pop = acs_pop.rename(columns={"B01003_001E": "total_population"})

acs_housing = clean_numeric(acs_housing, "B25001_001E")
acs_housing = acs_housing.rename(columns={"B25001_001E": "total_housing_units"})

acs_income = clean_numeric(acs_income, "B19013_001E")
acs_income = acs_income.rename(columns={"B19013_001E": "median_household_income"})

acs_value = clean_numeric(acs_value, "B25077_001E")
acs_value = acs_value.rename(columns={"B25077_001E": "median_home_value"})

print("ACS files loaded and cleaned.")

# ============================================================
# 3. MERGE ALL ACS VARIABLES
# ============================================================

acs_combined = acs_pop[["GEO_ID", "total_population"]] \
    .merge(acs_housing[["GEO_ID", "total_housing_units"]], on="GEO_ID", how="left") \
    .merge(acs_income[["GEO_ID", "median_household_income"]], on="GEO_ID", how="left") \
    .merge(acs_value[["GEO_ID", "median_home_value"]], on="GEO_ID", how="left")

print(f"Combined ACS data shape: {acs_combined.shape}")

# ============================================================
# 4. JOIN ACS DATA WITH TRACT GEOMETRIES
# ============================================================

# Extract tract GEOID from ACS GEO_ID (last 11 characters)
acs_combined["GEOID"] = acs_combined["GEO_ID"].str[-11:]

# Merge with tract geometries
tracts_acs = tracts.merge(
    acs_combined[["GEOID", "total_population", "total_housing_units", 
                  "median_household_income", "median_home_value"]],
    on="GEOID",
    how="left"
)

print(f"Final joined dataset shape: {tracts_acs.shape}")

# ============================================================
# 5. SAVE CLEANED DATASET
# ============================================================

output_path = "data/processed/tracts_with_acs_2024.parquet"
tracts_acs.to_parquet(output_path, index=False)

print(f"✅ Cleaned file saved to: {output_path}")
print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Cleaned ACS + Tract Join (with proper data types)
Using FIPS code '19' for input 'IA'
Using FIPS code '40' for input 'OK'
Using FIPS code '48' for input 'TX'
Total census tracts loaded: 8,985
ACS files loaded and cleaned.
Combined ACS data shape: (8998, 5)
Final joined dataset shape: (8985, 18)
✅ Cleaned file saved to: data/processed/tracts_with_acs_2024.parquet


In [14]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

file_path = "data/processed/tracts_with_acs_2024.parquet"

print("=" * 70)
print("Inspecting Cleaned ACS + Tract Dataset")
print("=" * 70)

# ============================================================
# 1. BASIC INFORMATION
# ============================================================

print("\n--- 1. Basic Information ---")
con.sql(f"""
    SELECT 
        COUNT(*) AS total_tracts,
        COUNT(*) FILTER (WHERE total_population IS NOT NULL) AS tracts_with_population
    FROM '{file_path}'
""").show()

# ============================================================
# 2. COLUMN NAMES AND DATA TYPES
# ============================================================

print("\n--- 2. Column Names and Data Types ---")
con.sql(f"""
    DESCRIBE '{file_path}'
""").show()

# ============================================================
# 3. SUMMARY STATISTICS
# ============================================================

print("\n--- 3. Summary Statistics ---")
con.sql(f"""
    SELECT 
        MIN(total_population) AS min_pop,
        MAX(total_population) AS max_pop,
        ROUND(AVG(total_population), 0) AS avg_pop,

        MIN(total_housing_units) AS min_housing,
        MAX(total_housing_units) AS max_housing,
        ROUND(AVG(total_housing_units), 0) AS avg_housing,

        MIN(median_household_income) AS min_income,
        MAX(median_household_income) AS max_income,
        ROUND(AVG(median_household_income), 0) AS avg_income,

        MIN(median_home_value) AS min_home_value,
        MAX(median_home_value) AS max_home_value,
        ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
""").show()

# ============================================================
# 4. NULL VALUE CHECK
# ============================================================

print("\n--- 4. Null Value Count ---")
con.sql(f"""
    SELECT 
        COUNT(*) - COUNT(total_population) AS null_population,
        COUNT(*) - COUNT(median_household_income) AS null_income,
        COUNT(*) - COUNT(median_home_value) AS null_home_value
    FROM '{file_path}'
""").show()

# ============================================================
# 5. SAMPLE DATA
# ============================================================

print("\n--- 5. Sample Rows ---")
con.sql(f"""
    SELECT 
        GEOID,
        NAME,
        total_population,
        total_housing_units,
        median_household_income,
        median_home_value
    FROM '{file_path}'
    LIMIT 8
""").show()

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Inspecting Cleaned ACS + Tract Dataset

--- 1. Basic Information ---
┌──────────────┬────────────────────────┐
│ total_tracts │ tracts_with_population │
│    int64     │         int64          │
├──────────────┼────────────────────────┤
│         8985 │                   8985 │
└──────────────┴────────────────────────┘


--- 2. Column Names and Data Types ---
┌─────────────────────────┬───────────────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │      column_type      │  null   │   key   │ default │  extra  │
│         varchar         │        varchar        │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼───────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ STATEFP                 │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ COUNTYFP                │ VARCHAR             